In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [3]:
%%capture
! pip uninstall unsloth unsloth_zoo -y
! pip install git+https://github.com/unslothai/unsloth-zoo.git --no-deps
! pip install git+https://github.com/unslothai/unsloth.git --no-deps

In [4]:
import os
# Note that to get best performance on A100, we needed to install causal-conv1d
# For this to not take too long, we had to downgrade to torch 2.8 (from torch 2.9)
# This means we wouldn't be able to use torch's grouped_mm here
# so we fallback to unsloth triton kernels for MoE
# We need to disable autotuning to save both time and memory for the colab notebook.
# If you are trying this elsewhere, we might recommend
# you install Flash Attention, Flash Linear Attention and CausalConv1d with torch 2.9
# `!uv pip install --no-build-isolation flash-attn flash-linear-attention causal_conv1d==1.6.`
# You can even try playing around with the below env var for faster performance but make sure you have enough VRAM to try autotuning.
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE'] = '1'

In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, processor = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-4B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False,
)
tokenizer = processor.tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [6]:
import json
from pathlib import Path

sentences = []
y = []

with open(r'/content/drive/MyDrive/annotation_edited/shared_data.json', 'r', encoding = 'utf-8') as f:
  data = json.load(f)
  sentences = data['content']
  y = data['labels']
print(len(sentences), len(y))

13281 13281


In [15]:
FastLanguageModel.for_inference(model)
emotions = ['contentment', 'anger', 'amusement', 'awe', 'disgust', 'excitement', 'fear', 'sadness']

def prompt(emotions, sentence):
  return f'Choose one emotion from the list below that best describes the sentence.\nList: {emotions} \n\n Sentence: {sentence} \n Emotion:'

limit = 100
results = []

for sentence in sentences[:limit]:

  inputs = tokenizer.apply_chat_template(
      [
          {"role": "system", "content": "You are a classifier. Respond ONLY with one word from the list, providing the emotion."},
          {"role" : "user", "content" : prompt(emotions, sentence)}
      ],
      tokenize = True,
      add_generation_prompt = True,
      return_tensors = "pt",
      enable_thinking = False,
      return_dict = True,
  ).to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens = 15,
      use_cache = True,
      temperature = 0.1,
      tokenizer = tokenizer
  )

  # results.append(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split('\n\n')[-1][:-1])
  results.append(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])
print(results)

["system\nYou are a classifier. Respond ONLY with one word from the list, providing the emotion.\nuser\nChoose one emotion from the list below that best describes the sentence.\nList: ['contentment', 'anger', 'amusement', 'awe', 'disgust', 'excitement', 'fear', 'sadness'] \n\n Sentence: The video shows a man with red hair wearing a blue shirt, aggressively standing in front of a chain link fence. There is another man in a black beanie and white shirt standing behind him. The background includes trees and a building. \n Emotion:\nassistant\n<think>\n\n</think>\n\nanger\n", 'system\nYou are a classifier. Respond ONLY with one word from the list, providing the emotion.\nuser\nChoose one emotion from the list below that best describes the sentence.\nList: [\'contentment\', \'anger\', \'amusement\', \'awe\', \'disgust\', \'excitement\', \'fear\', \'sadness\'] \n\n Sentence: In the video, a man in a blue shirt and black pants is confrontationally talking to another man in a black beanie and 

In [16]:
from sklearn.metrics import classification_report
results = [x.split('\n\n')[-1][:-1] for x in results]
print(classification_report(y[:limit], results))

              precision    recall  f1-score   support

   amusement       0.20      0.25      0.22         4
       anger       1.00      0.22      0.36        37
         awe       0.33      0.75      0.46         4
 contentment       0.26      1.00      0.41         6
     disgust       1.00      0.67      0.80         6
  excitement       1.00      0.17      0.29         6
        fear       0.48      0.78      0.60        18
     sadness       0.71      0.79      0.75        19

    accuracy                           0.52       100
   macro avg       0.62      0.58      0.49       100
weighted avg       0.75      0.52      0.50       100



In [17]:
import torch

FastLanguageModel.for_inference(model)
model.eval()

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

emotions = [
    "contentment", "anger", "amusement", "awe",
    "disgust", "excitement", "fear", "sadness"
]

emotion_ids = torch.tensor(
    [tokenizer.encode(e, add_special_tokens=False)[0] for e in emotions],
    device="cuda"
)

def make_text(sentence):
    return f"Choose one emotion: {', '.join(emotions)}.\nSentence: {sentence}\nEmotion:"

results = []
batch_size = 1

for i in range(0, len(sentences[:limit]), batch_size):
    batch = sentences[i:i + batch_size]
    texts = [make_text(s) for s in batch]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    pred = logits[:, emotion_ids].argmax(dim=-1).tolist()
    results.extend([emotions[j] for j in pred])

In [18]:
print(classification_report(y[:limit], results))

              precision    recall  f1-score   support

   amusement       0.00      0.00      0.00         4
       anger       0.75      0.49      0.59        37
         awe       1.00      0.50      0.67         4
 contentment       0.08      1.00      0.15         6
     disgust       0.00      0.00      0.00         6
  excitement       0.00      0.00      0.00         6
        fear       0.00      0.00      0.00        18
     sadness       0.50      0.05      0.10        19

    accuracy                           0.27       100
   macro avg       0.29      0.25      0.19       100
weighted avg       0.42      0.27      0.27       100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
